*FORECASTING*

eksplorasi data serta membuat alat prediksi curah hujan untuk 3 hari yang akan mendatang/lebih

In [ ]:
import pandas as pd
import re

In [ ]:
file19 = 'C:/Users/nalia-pc/Dropbox/PC/Documents/===KULIAH===/PKL/F2019.xlsx'
file20 = 'C:/Users/nalia-pc/Dropbox/PC/Documents/===KULIAH===/PKL/F2020.xlsx'
file21 = 'C:/Users/nalia-pc/Dropbox/PC/Documents/===KULIAH===/PKL/F2021.xls'
files = [file19, file20, file21]
# y = ['19', '20', '21']
m = ['JAN', 'FEB', 'MAR', 'APR', 'MEI', 'JUN', 'JUL', 'AGS', 'SEPT', 'OKT', 'NOV', 'DES']
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
years = ['2019', '2020', '2021']

Details :

- tgl         : tanggal
- temp*       : suhu pada jam * WIB
- temp_avg    : rata2 suhu
- temp_24     : rata2 suhu 24 jam
- temp_max    : suhu max
- temp_min    : suhu min
- CH          : Curah hujan dalam mm jam 7 WIB
- light_hour  : lama penyinaran matahari dalam jam (08.00-16.00)
- light_per   : lama penyinaran matahari (%) (08.00-16.00)
- pck         : peristiwa cuaca khusus
----------------------------------------------------
- tgl           : tanggal
- press         : tekanan udara (mb)
- humid*        : lembab NISBI (%)
- humid_avg     : rata-rata kelembapan
- humid_24     : rata-rata kelembapan 24j
- ws_avg        : kecepatan rata2 angin (Knot)
- mod_dir       : arah terbanyak
- max_ws        : kecepatan angin terbesar (Knot)
- dir           : arah angin

In [ ]:
xl = pd.ExcelFile('F2019.xlsx')

regex = re.compile('JAN')

sheets = [n for n in xl.sheet_names if regex.match(n)]
# ['matching_sheet1', 'matching_sheet2']

dfs = pd.read_excel(xl, sheet_name=sheets)
key = list(dfs.keys())[0]
print(key)

In [ ]:
df = pd.DataFrame()
y_dex = 0
for i in files:
    x_dex = 0
    xl = pd.ExcelFile(i)
    for month in months:
        regex = re.compile(m[month-1])
        sheets = [n for n in xl.sheet_names if regex.match(n)]
        dic = pd.read_excel(xl, sheet_name=sheets)
        sheet_k = list(dic.keys())[0]
        
        data_1 = pd.read_excel(i,
                                sheet_name= sheet_k,
                                header = 19,
                                nrows = 33,
                                usecols = 'A:L',
                                names=['tgl', 'temp7', 'temp13', 'temp18', 'temp_avg','temp_24', 'temp_max', 'temp_min', 'CH', 'light_hour', 'light_per', 'pck']
                                )
        data_2 = pd.read_excel(i,
                                sheet_name= sheet_k,
                                header = 6,
                                nrows = 33,
                                usecols = 'M:U,W:X',
                                names=['tgl', 'press', 'humid7', 'humid13', 'humid18', 'humid_avg', 'humid_24', 'ws_abg', 'mod_dir', 'max_ws', 'dir']
                                )
       
        # datas = pd.merge(data_1, data_2, on='tgl', how='left')
        datas = pd.concat([data_1, data_2], axis=1, join='inner')
        datas.dropna(axis = 0, subset=['tgl'], inplace = True)
        datas.insert(0, 'Bulan', str(month))
        datas.insert(0, 'Tahun', str(years[y_dex]))
        print(datas.tail())
        df = pd.concat([df,datas], ignore_index=True)
        x_dex += 1
    y_dex += 1


In [ ]:
df.isna().sum()

In [ ]:
df

In [ ]:
# save structured data to csv

# df.to_csv('Clean_Weather.csv', index=False, encoding='utf-8-sig')

-------------------------------------------------------------------------------------------------------------------------
# Data Cleaning

- Data Kosong dalam dataframe menunjukkan ketidakhadiran variabel pada hari tersebut.

- TTU merupakan keadaan dimana hujan terjadi namun dikarenakan sangat kecil maka tidak dapat terukur, sehingga perlu didiskusikan lagi apakah perlu dituliskan '0' atau nilai yang sangat kecil seperti '0.01'

- tanda '-' menandakan bahwa tidak adanya hujan sehingga akan dikonversi ke nilai '0'

- pck memiliki beberapa data yang kosong yang berarti tidak adanya kejadian khsusu pada hari tersebut.

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/weather-prediction/Clean_Weather.csv')
df.head()

In [ ]:
df.isna().sum()

In [ ]:
display(df.iloc[845])

In [ ]:
i = df[(df.tgl == ' ')].index
df = df.drop(i)
df = df.fillna(0)

In [ ]:
col_list = ['-', 'TTU']
for i in col_list:
    count = df.CH.str.contains(i).sum()
    print(i, ':', count)

In [ ]:
df.loc[df['CH'] == '-', 'CH'] = 0
df.loc[df['CH'] == 'TTU', 'CH'] = 0.01

In [ ]:
col_list = ['-', 'TTU']
for i in col_list:
    count = df.CH.str.contains(i).sum()
    print(i, ':', count)

In [ ]:
df.isna().sum()

In [ ]:
cols = ['tgl', 'temp7', 'temp13', 'temp18', 'temp_avg','temp_24', 'temp_max', 'temp_min', 'CH', 'light_hour', 'light_per', 'press', 'humid7', 'humid13', 'humid18', 'humid_avg', 'humid_24', 'ws_abg', 'max_ws', 'dir']
for i in cols:    
    df[i] = df[i].astype(str).astype(float)

In [ ]:
df.dtypes

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import numpy as np
import seaborn as sns

In [ ]:
df['CH'].describe()

In [ ]:
dff = df.groupby(["Tahun"]).CH.describe().reset_index()

In [ ]:
dff

Total Curah Hujan per tahun

In [ ]:
dff = df.groupby(["Tahun"]).CH.sum().reset_index()
plt.figure(figsize=(8,4), tight_layout=True)
ax = sns.barplot(x=dff['Tahun'], y=dff['CH'], palette='pastel', ci=None)
ax.set(title='Total Curah Hujan per Hari 2019-2021', xlabel='Tahun', ylabel='Curah Hujan')
plt.show()

In [ ]:
df['date'] = pd.to_datetime(dict(year=df.Tahun, month=df.Bulan, day=df.tgl))

In [ ]:
df.head()

In [ ]:
#Timeseries plot 2019-2021

plt.figure(figsize=(25,6))
plt.plot(df["date"], df["CH"])
plt.xlabel("Date")
plt.ylabel("CH")
plt.show()

In [ ]:
#Timeseries plot yearly
df19 = df.loc[df['Tahun'] == 2019]
df20 = df.loc[df['Tahun'] == 2020]
df21 = df.loc[df['Tahun'] == 2021]
all_df = [df19, df20, df21]
# create line plot of sales data
for i in all_df:
    plt.figure(figsize=(25,6))
    plt.plot(i["date"], i["CH"])
    plt.xlabel("Date")
    plt.ylabel("Curah Hujan")
    plt.show()

In [ ]:
import seaborn as sns

#Timeseries plot yearly
df19 = df.loc[df['Tahun'] == 2019]
df20 = df.loc[df['Tahun'] == 2020]
df21 = df.loc[df['Tahun'] == 2021]
all_df = [df19, df20, df21]
# create line plot of sales data
for i in all_df:
    fig, ax = plt.subplots(figsize=(20,5))
    pl = sns.barplot(data=i, x='Bulan', y="CH",ci=None, 
                ax = ax)
    pl.set(title='Total Curah Hujan per Bulan', xlabel=i['Tahun'].iloc[0], ylabel='Curah Hujan')

# Correlation between variables

In [ ]:
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder

In [ ]:
### direction of the wind
encoder = OneHotEncoder(handle_unknown='ignore')

#perform one-hot encoding on 'team' column 
encoder_df = pd.DataFrame(encoder.fit_transform(df[['mod_dir']]).toarray())
encoder_df.columns = encoder.get_feature_names_out()

#merge one-hot encoded columns back with original DataFrame
df2 = pd.concat([df.reset_index(drop=True), encoder_df], axis='columns')

#view final df
print(df2)

## 

In [ ]:
list = df['pck'].values.tolist()
list[0]

In [ ]:
df['pck']=df['pck'].replace([list[0],0],'None')
### PCK
encoder_df2 = pd.DataFrame(encoder.fit_transform(df[['pck']]).toarray())
encoder_df2.columns = encoder.get_feature_names_out()
encoder_df2.columns = encoder_df2.columns.str.replace(' ', '')

#merge one-hot encoded columns back with original DataFrame
df2 = pd.concat([df2.reset_index(drop=True), encoder_df2], axis='columns')

#view final df
print(df2)

In [ ]:
corr_df2 = df2.drop(['Tahun', 'tgl', 'tgl.1', 'date'], axis=1)

In [ ]:
fig, size = plt.subplots(figsize=(20,10)) 
ax = sns.heatmap(corr_df2.corr(method="spearman"), annot=True, cmap="crest", ax=size) #spearman
ax.set(xlabel="", ylabel="")
ax.xaxis.tick_top()

In [ ]:
corr_CH = pd.DataFrame(corr_df2.corrwith(corr_df2["CH"], method='spearman').sort_values())
corr_CH

In [ ]:
# 0.3 is used for illustration 
# replace with your actual value
thresh = 0.4

mask = corr_CH.abs() > thresh
# or mask = coeff < thresh

corr_CH.where(mask).stack()

# Forecasting

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

## CH besok

In [ ]:
#Curah Hujan Besok
df3 = df2
df3['ch_besok'] = df3.shift(-1)['CH']
df3 = df3.iloc[:-1,:].copy()

## CH lusa

In [ ]:
# df3['ch_besok2'] = df3.shift(-2)['CH']
# df3 = df3.iloc[:-2,:].copy()

## CH 3 hari kedepan

In [ ]:
# df3['ch_besok3'] = df3.shift(-3)['CH']
# df3 = df3.iloc[:-3,:].copy()

In [ ]:
df3

In [ ]:
corr_CH = pd.DataFrame(df3.corrwith(df3["ch_besok"], method='spearman').sort_values())
thresh = 0.5
mask = corr_CH.abs() > thresh
corr_CH.where(mask).stack()

In [ ]:
# corr_CH = pd.DataFrame(df3.corrwith(df3["ch_besok2"], method='spearman').sort_values())
# thresh = 0.4
# mask = corr_CH.abs() > thresh
# corr_CH.where(mask).stack()

In [ ]:
# corr_CH = pd.DataFrame(df3.corrwith(df3["ch_besok3"], method='spearman').sort_values())
# thresh = 0.4
# mask = corr_CH.abs() > thresh
# corr_CH.where(mask).stack()

In [ ]:
#create train and test dataset
split_date = pd.datetime(2021,6 ,1)

# df_training = df.loc[df['Date'] <= split_date]
# df_test = df.loc[df['Date'] > split_date]

train = df3.loc[df3['date'] < split_date]
test = df3.loc[df3['date'] >= split_date]

# fig, ax = plt.subplots(figsize=(15,5))
# train.plot()
# test.plot()
# plt.show()

In [ ]:
df3.columns

In [ ]:
# fitur = ['pck_None', 'light_per', 'light_hour', 'pck_TSRA','humid13', 'humid18', 'pck_RA', 'humid7', 'humid_24', 'humid_avg' ]
# fitur = ['pck_None', 'light_per', 'light_hour','temp_24', 'temp18', 'mod_dir_TG', 'humid7', 'CH', 'pck_TSRA','humid13', 'humid18', 'humid_24', 'humid_avg' ]
fitur = ['pck_None', 'pck_TSRA','humid13', 'humid18', 'pck_RA', 'humid7', 'humid_24', 'humid_avg' ]
""" 
pck_None    0   -0.601333
light_per   0   -0.502403
light_hour  0   -0.502403
temp_24     0   -0.430938
temp18      0   -0.404957
mod_dir_TG  0   -0.402750
humid7      0    0.436156
CH          0    0.506746
pck_TSRA    0    0.519518
humid13     0    0.543345
humid_avg   0    0.623953
humid18     0    0.632478
humid_24    0    0.707581
ch_besok    0    1.000000


From the correlation we have calculated :
for corr >0.4
pck_None    0   -0.826615
light_per   0   -0.435565
light_hour  0   -0.435565
pck_TSRA    0    0.487197
humid13     0    0.527012
humid18     0    0.531523
pck_RA      0    0.616398
humid7      0    0.620337
humid_24    0    0.622988
humid_avg   0    0.634803
CH          0    1.000000

if >0.5

pck_None   0   -0.826615
humid13    0    0.527012
humid18    0    0.531523
pck_RA     0    0.616398
humid7     0    0.620337
humid_24   0    0.622988
humid_avg  0    0.634803
CH         0    1.000000
"""

target = 'ch_besok'
# target2 = 'ch_besok2'
# target3 = 'ch_besok3'

In [ ]:
X_train = train[fitur]
y_train = train[target]

# y_train2 = train[target2]
# y_train3 = train[target3]


X_test = test[fitur]
y_test = test[target]

# y_test2 = test[target2]
# y_test3 = test[target3]

## Continue to Prediction

In [ ]:
# pembentukan prediksi model dengan xgboost. squared error memberikan hasil yang lebih ekstrim. pseudohubererror memberikan hasil yang lebih aman namun tidak dapat memberikan nilai tinggi ketika prediksi curah hujan yang sebenarnya tinggi.
# reg = xgb.XGBRegressor(objective='reg:pseudohubererror', n_estimators=1000, early_stopping_rounds=50)

reg = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, early_stopping_rounds=50)

reg.fit(X_train, y_train,
       eval_set=[(X_train, y_train), (X_test, y_test)],
       verbose=True)

In [ ]:
# reg2 = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, early_stopping_rounds=50)

# reg2.fit(X_train, y_train2,
#        eval_set=[(X_train, y_train2), (X_test, y_test2)],
#        verbose=True)

In [ ]:
# reg3 = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, early_stopping_rounds=50)

# reg3.fit(X_train, y_train3,
#        eval_set=[(X_train, y_train3), (X_test, y_test3)],
#        verbose=True)

# Feature Importance

In [ ]:
fi = pd.DataFrame(data=reg.feature_importances_,
index=reg.feature_names_in_,
columns=['importance'])

# fi2 = pd.DataFrame(data=reg2.feature_importances_,
# index=reg2.feature_names_in_,
# columns=['importance'])

# fi3 = pd.DataFrame(data=reg3.feature_importances_,
# index=reg3.feature_names_in_,
# columns=['importance'])

In [ ]:
fi.sort_values('importance').plot(kind='barh', title='Feature Importance')
plt.show()

# fi2.sort_values('importance').plot(kind='barh', title='Feature Importance')
# plt.show()

# fi3.sort_values('importance').plot(kind='barh', title='Feature Importance')
# plt.show()

# Forecast on Test

In [ ]:
test['prediksi'] = reg.predict(X_test)
df3_pred = df3.merge(test[['prediksi']], how='left', left_index=True, right_index=True)

# test['prediksi2'] = reg2.predict(X_test)
# df3_pred = df3_pred.merge(test[['prediksi2']], how='left', left_index=True, right_index=True)

# test['prediksi3'] = reg3.predict(X_test)
# df3_pred = df3_pred.merge(test[['prediksi3']], how='left', left_index=True, right_index=True)

In [ ]:
df3_pred

In [ ]:
# ax = df3_pred[['CH']].plot(figsize=(15, 5))
# df3_pred['prediksi'].plot(ax=ax)
# plt.legend(['Data sebenarnya','prediksi'])
# ax.set_title('Perbandingan prediksi dengan data sebenarnya')
# plt.show()

In [ ]:
split_date = pd.datetime(2021,6,1)

# df_training = df.loc[df['Date'] <= split_date]
# df_test = df.loc[df['Date'] > split_date]
df3_pred[['ch_besok', 'prediksi']].loc[df3_pred['date'] > split_date].plot(figsize=(15, 5), title = 'Prediksi Curah Hujan +1 hari')
# df3_pred[['ch_besok', 'prediksi2']].loc[df3_pred['date'] > split_date].plot(figsize=(15, 5), title = 'Prediksi Curah Hujan +2 hari')
# df3_pred[['ch_besok', 'prediksi3']].loc[df3_pred['date'] > split_date].plot(figsize=(15, 5), title = 'Prediksi Curah Hujan +3 hari')
plt.show()

In [ ]:
df3['CH'].max()

In [ ]:
y_pred = test['prediksi']
from sklearn.metrics import r2_score
r2_score(y_test, test['prediksi'])

#.tail(183)

In [ ]:
import numpy as np
mape = np.mean(np.abs(y_pred - y_test)/np.abs(y_test))
mae = np.mean(np.abs(y_pred - y_test))
mpe = np.mean((y_pred - y_test)/y_test)
rmse = np.mean((y_pred - y_test)**2)**0.5
corr = np.corrcoef(y_pred, y_test)[0,1]

mins = np.amin(np.hstack([y_pred[:,None], y_test[:,None]]), axis=1)
maxs = np.amax(np.hstack([y_pred[:,None], y_test[:,None]]), axis=1)
minmax = 1-np.mean(mins/maxs)

import pprint
pprint.pprint({'mape' : mape, 'mae':mae,
              'mpe':mpe, 'rmse':rmse,
              'corr':corr, 'minmax':minmax})

In [ ]:
#MASE (cek kembali apakah bisa digunakan untuk forecasting selain ARIMA)
n = np.array(y_test).shape[0]
d = np.abs(np.diff(np.array(y_test))).sum()/(n-1)

errors = np.abs(y_test - y_pred)
print(errors.mean()/d)

Note : coba menggunakan Kalman FIlter https://medium.com/data-science-in-your-pocket/time-series-forecasting-using-kalman-filter-814e9171d780

# Menggunakan model ARIMA

In [ ]:

from statsmodels.tsa.stattools import adfuller
def ad_test(dataset):
    dftest = adfuller(dataset, autolag = 'AIC')
    print("1. ADF : ", dftest[0])
    print("2. P-Value : ", dftest[1])
    print("3. Num Of Lags : ", dftest[2])
    print("4. Num Of Observations Used For ADF Regression and Critical Values Calculation :", dftest[3])
    print("5. Critical Values :")
    for key, val in dftest[4].items():
        print("\t", key, ": ", val)

In [ ]:
ad_test(df2['CH'])

In [ ]:
rolling_mean = data.rolling(window = 12).mean()
data['rolling_mean_diff'] = rolling_mean - rolling_mean.shift()
ax1 = plt.subplot()
data['rolling_mean_diff'].plot(title='after rolling mean & differencing');
ax2 = plt.subplot()
data.plot(title='original');

Hasil menyatakan bahwa data stasioner (0.0002 < 0.05)

## Arima Model

In [ ]:
pip install pmdarima

In [ ]:
import pmdarima
from pmdarima import auto_arima
import warnings
warnings.filterwarnings("ignore")

In [ ]:
dfar = df2
dfar['ch_besok'] = dfar.shift(-1)['CH']
dfar = dfar.iloc[:-1,:].copy()

In [ ]:
fitur = ['date', 'pck_None', 'light_per', 'light_hour', 'pck_TSRA','humid13', 'humid18', 'pck_RA', 'humid7', 'humid_24', 'humid_avg', 'ch_besok' ]
dfar = dfar[fitur]
stepwise_fit = auto_arima(dfar['ch_besok'], trace=True,
                         suppress_warnings=True)
stepwise_fit.summary()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
#create train and test dataset
split_date = pd.datetime(2021,6 ,1)

train = dfar.loc[dfar['date'] < split_date]
test = dfar.loc[dfar['date'] >= split_date]


In [ ]:
train['ch_besok']

In [ ]:
model=ARIMA(dfar['ch_besok'], order=(4,0,1))
history = model.fit()
history.summary()
    

In [ ]:
# import statsmodels.api as sm
# model = sm.tsa.statespace.SARIMAX(train['ch_besok'], order=(1,1,1))
# model = model.fit()
# model.summary()

In [ ]:
start = len(train)
end=len(train)+len(test)-1
pred=history.predict(start=start, end=end, typ = 'levels')
pred

In [ ]:
# dfar_pred = test.merge(pred, how='left', left_index=True, right_index=True)

In [ ]:
test['ch_besok'].plot(legend=True)
pred.plot(legend=True)


In [ ]:
test['ch_besok'].mean()

In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt
rmse = sqrt(mean_squared_error(pred,test['ch_besok']))
print(rmse)

Hasil ARIMA kurang bagus. model perlu diperbaiki. Hal ini diduga dikarenakan banyaknya outlier.